# PT-RL-SYSTEM. Rollout, Training and Reward System | 采样、训练与奖励系统

**难度：** Hard | **环境：** CPU-first | **标签：** `后训练`, `RL System`, `Resource Scheduling`

---

## 本节导读

在线后训练同时运行 rollout、奖励计算和 learner 更新。系统吞吐不由单个组件的峰值决定，而由最慢阶段、队列增长、权重同步和失败恢复共同决定。

本节消费已经完成身份与版本检查的轨迹，比较共置与分离部署，并把阶段处理率、资源池驻留、跨池载荷和同步门槛收敛为系统拓扑准入结论。

**关键词：** `rollout worker`, `learner`, `colocation`, `disaggregation`, `pipeline bottleneck`

![后训练系统角色与数据流](../docs/public/02_PyTorch_Algorithms/pt_rl_system_topology.svg)

## 前置阅读

**导语：** 先理解轨迹身份和策略同步，再把算法角色放入多阶段系统。

- [Part 02 · PT-ROLLOUT-LIFECYCLE Rollout 与策略新鲜度](./PT-ROLLOUT-LIFECYCLE_Rollout_and_Policy_Freshness.ipynb)
- [Part 02 · 16 PPO 资源与状态](./16_RLHF_PPO_Memory.ipynb)

### Step 1：先区分五类系统角色

角色可以共享 GPU，也可以拆到不同资源池。选择拓扑前先明确每个角色持有的状态、单位时间处理量和输出对象，否则无法比较队列增长、显存驻留和跨池传输。

| 角色 | 持有状态 | 主要输出 |
| --- | --- | --- |
| rollout worker | 当前 policy、KV Cache、采样状态 | response、logprobs、policy version |
| reward / verifier | Reward Model 或规则 | reward、验证状态 |
| reference worker | 冻结 reference policy | reference logprobs |
| learner | policy、optimizer、梯度与训练状态 | 新 checkpoint / 权重版本 |
| artifact / coordinator | 版本、队列和运行状态 | 分发、确认、失败与恢复记录 |

### Step 2：共置与分离部署改变驻留、争用和传输

| 候选 | 优势 | 主要代价 | 首先验证 |
| --- | --- | --- | --- |
| 共置 | 少跨池传输、部署简单 | 多模型争用显存，阶段切换可能空转 | 各资源池峰值显存、切换时间 |
| 分离 | 角色独立扩缩容、故障隔离 | 轨迹与权重传输、队列和版本同步 | 每秒载荷、传输时间、策略滞后 |
| 混合 | 高频交互角色共置，重模型拆分 | 配置与调度复杂 | 每条跨池边的暴露传输时间 |

### Step 3：吞吐由瓶颈阶段和同步门槛共同限制

先把采样、奖励和学习阶段的处理率统一为样本每秒。流水线容量由最小处理率决定；并列最小值表示多个阶段同时构成瓶颈。对阶段 `i`，队列余量为 `r_i - arrival_rate`，任一余量为负都会导致队列持续增长。

跨池边还要把单样本载荷乘以传输频率，再除以有效带宽得到传输占用。可重叠部分不等于消失，只有未被计算覆盖的暴露传输时间进入关键路径。拓扑准入因此需要同时检查队列稳定性、各资源池显存、暴露传输、版本确认和恢复能力。

![候选拓扑的准入与瓶颈判断](../docs/public/02_PyTorch_Algorithms/pt_rl_system_decision.svg)

### Step 4：实现系统账本与候选准入

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：定位流水线瓶颈<br>`pipeline_capacity` | 计算容量、并列瓶颈和各阶段队列余量 | 所有速率使用样本每秒且必须为正 | 单瓶颈、并列瓶颈、稳定与增长队列 |
| TODO 2：建立跨池传输账本<br>`transfer_ledger` | 将单样本载荷和频率换算为传输与暴露时间 | 同池边不产生网络传输，重叠比例属于 0–1 | 同池、跨池、带宽与重叠变化 |
| TODO 3：执行拓扑准入<br>`evaluate_topology` | 联合队列、分池显存、暴露传输、版本和恢复门槛 | 硬门槛失败时拒绝，容量或传输不足时调优 | 可行、队列增长、分池超限、同步与恢复失败 |

In [ ]:
from typing import Dict, List


In [ ]:
def pipeline_capacity(stage_rates, arrival_rate):
    """返回稳态容量、并列瓶颈与各阶段队列余量。"""
    # ==========================================
    # TODO 1：按统一的样本每秒速率判断流水线是否稳定。
    # capacity = ???  # 所有阶段处理率的最小值。
    # queue_margins = ???  # 各阶段处理率减到达率。
    # ==========================================
    raise NotImplementedError("TODO 1：计算流水线容量与队列余量")

def transfer_ledger(edges, placement):
    """汇总跨资源池 payload；同池边不产生网络传输。"""
    # ==========================================
    # TODO 2：计算每条跨池边的载荷率、传输时间和暴露时间。
    # traffic_bytes_per_s = ???  # 单样本字节数乘每秒样本数。
    # exposed_seconds_per_s = ???  # 未被重叠隐藏的传输占用。
    # ==========================================
    raise NotImplementedError("TODO 2：建立跨池传输账本")

def evaluate_topology(capacity_report, memory_by_pool, memory_limits, transfer_report, version_sync_ok, recovery_ok, max_exposed_s_per_s=0.1):
    """返回 accept/tune/reject 及可复核原因。"""
    # ==========================================
    # TODO 3：依次检查分池显存、版本、恢复、队列和暴露传输。
    # over_limit_pools = ???  # 驻留量超过预算的资源池。
    # decision = ???  # accept、tune 或 reject。
    # ==========================================
    raise NotImplementedError("TODO 3：执行系统拓扑准入")

### 测试

测试分别验证瓶颈归因、跨池载荷和系统准入。

In [ ]:
def test_capacity():
    """稳态容量必须由最慢阶段决定。"""
    result = pipeline_capacity({'rollout': 120, 'reward': 80, 'learner': 80}, arrival_rate=70)
    assert result['capacity'] == 80
    assert result['bottlenecks'] == ['learner', 'reward']
    assert result['queue_stable'] is True
    assert pipeline_capacity({'rollout': 120, 'reward': 80}, 90)['queue_stable'] is False

def test_transfer():
    """只有跨资源池的边才能计入传输账本。"""
    edges = [
        {'source': 'rollout', 'target': 'reward', 'bytes_per_item': 100, 'items_per_second': 10, 'bandwidth_bytes_per_s': 1000, 'overlap_fraction': 0.5},
        {'source': 'learner', 'target': 'rollout', 'bytes_per_item': 1000, 'items_per_second': 1, 'bandwidth_bytes_per_s': 2000, 'overlap_fraction': 0.0},
    ]
    report = transfer_ledger(edges, {'rollout': 'A', 'reward': 'A', 'learner': 'B'})
    assert report['cross_pool_bytes_per_s'] == 1000
    assert report['exposed_seconds_per_s'] == 0.5

def test_gate():
    """容量不足返回 tune，显存、同步或恢复失败返回 reject。"""
    stable = pipeline_capacity({'rollout': 120, 'reward': 80}, 70)
    unstable = pipeline_capacity({'rollout': 120, 'reward': 80}, 90)
    low_transfer = {'exposed_seconds_per_s': 0.05}
    assert evaluate_topology(stable, {'A': 30}, {'A': 40}, low_transfer, True, True)['decision'] == 'accept'
    assert evaluate_topology(unstable, {'A': 30}, {'A': 40}, low_transfer, True, True)['decision'] == 'tune'
    assert evaluate_topology(stable, {'A': 50}, {'A': 40}, low_transfer, True, True)['decision'] == 'reject'
    assert evaluate_topology(stable, {'A': 30}, {'A': 40}, low_transfer, False, True)['decision'] == 'reject'

test_capacity(); test_transfer(); test_gate()
print('✅ 后训练系统机制测试通过')

---

🛑 **STOP HERE** 🛑

> 请先完成题目区并运行测试，再查看参考实现。

---

## 参考代码与解析

In [ ]:
def pipeline_capacity(stage_rates, arrival_rate):
    if not stage_rates or any(rate <= 0 for rate in stage_rates.values()):
        raise ValueError('stage rate 必须为正')
    if arrival_rate < 0:
        raise ValueError('arrival_rate 不能为负')
    capacity = min(stage_rates.values())
    bottlenecks = sorted(name for name, rate in stage_rates.items() if rate == capacity)
    queue_margins = {name: rate - arrival_rate for name, rate in stage_rates.items()}
    return {'capacity': capacity, 'bottlenecks': bottlenecks, 'queue_margins': queue_margins,
            'queue_stable': all(margin >= 0 for margin in queue_margins.values())}

def transfer_ledger(edges, placement):
    rows, total_bytes, total_exposed = [], 0.0, 0.0
    for edge in edges:
        cross_pool = placement[edge['source']] != placement[edge['target']]
        overlap = float(edge.get('overlap_fraction', 0.0))
        bandwidth = float(edge['bandwidth_bytes_per_s'])
        if not 0.0 <= overlap <= 1.0 or bandwidth <= 0:
            raise ValueError('重叠比例须属于 0–1，带宽必须为正')
        traffic = float(edge['bytes_per_item']) * float(edge['items_per_second']) if cross_pool else 0.0
        transfer = traffic / bandwidth
        exposed = transfer * (1.0 - overlap)
        total_bytes += traffic
        total_exposed += exposed
        rows.append(edge | {'cross_pool': cross_pool, 'traffic_bytes_per_s': traffic,
                            'transfer_seconds_per_s': transfer, 'exposed_seconds_per_s': exposed})
    return {'cross_pool_bytes_per_s': total_bytes, 'exposed_seconds_per_s': total_exposed, 'rows': rows}

def evaluate_topology(capacity_report, memory_by_pool, memory_limits, transfer_report, version_sync_ok, recovery_ok, max_exposed_s_per_s=0.1):
    over_limit_pools = sorted(pool for pool, used in memory_by_pool.items() if used > memory_limits.get(pool, 0))
    if over_limit_pools:
        return {'decision': 'reject', 'reason': 'memory_limit', 'over_limit_pools': over_limit_pools}
    if not version_sync_ok:
        return {'decision': 'reject', 'reason': 'version_sync'}
    if not recovery_ok:
        return {'decision': 'reject', 'reason': 'recovery_not_ready'}
    if not capacity_report['queue_stable']:
        return {'decision': 'tune', 'reason': 'unstable_queue'}
    if transfer_report['exposed_seconds_per_s'] > max_exposed_s_per_s:
        return {'decision': 'tune', 'reason': 'exposed_transfer'}
    return {'decision': 'accept', 'reason': 'all_gates_passed'}

### 解析

#### TODO 1：计算流水线容量与队列余量

- 统一处理率单位后，最慢阶段决定容量；阶段处理率减到达率决定队列是否持续增长。

#### TODO 2：建立跨池传输账本

- 只统计资源池不同的边；载荷率除以带宽得到传输占用，再扣除可重叠部分得到暴露时间。

#### TODO 3：执行拓扑准入

- 分池显存、版本确认和恢复能力是硬门槛；队列增长或暴露传输过高进入调优。

## 相关阅读

- [Part 02 · 85 GRPO 组级对齐项目](./85_GRPO_Groupwise_Alignment_Project.ipynb)
- [Part 02 · 86 Online DPO Benchmark](./86_DPO_Online_Benchmark.ipynb)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)
- [TRL](https://github.com/huggingface/trl)